# Exercise 2. Classify Text with BOW and TF-IDF
As mentioned in this week's lecture, we can represent text as document matrices using either a `Bag-of-Words`(`BOW`) or `TF-IDF` vectorization.

To recap, `BOW` is a simple count of each `term` (or word) in each text document, creating simple vectors for each document:
```{figure} ../figures/class2/bow.png
---
name: bow
---
Figure from [Zhou (2019)](https://victorzhou.com/blog/bag-of-words/).
```

`TF-IDF` is similar to `BOW`, but instead of using raw counts, it weights each term by its frequency in the document and its inverse document frequency across the corpus. The inverse document frequency reduces the influence of terms that occur frequently across documents and increases the weight of terms that are rare. See also [geeksforgeeks.org](https://www.geeksforgeeks.org/machine-learning/understanding-tf-idf-term-frequency-inverse-document-frequency/).


```{admonition} LLM FRAMING: Why BOW/TF-IDF still matter in the age of LLMs!
:class: dropdown, fuchsia
There are many reasons to still care about classifiers relying on BOW and TF-IDF:
1. They tend to perform relatively well compared to a LLM classifier. 
2. It is far cheaper computationally to train than fine-tuning an LLM. 
3. Even when fine-tuning makes sense, we need baselines to measure progress, and BOW/TF-IDF are good for this! 

Our key question should be: Is an LLM-approach worth it if a simpler baseline works just as well?
```

In [ ]:
## CODE CHUNK REMOVED FOR USERS - HERE TO RELOAD DATA ##
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

## LOAD DATA ## 
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

raw_df = pd.read_csv(data_path)

## SUBSET DATA ##
df = raw_df[raw_df["model"].isin(["human", "chatgpt"])]

df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

## SPLIT DATA ##
train_df, val_df= train_test_split(
                                                    df,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=df["is_human"]
                                                    )

## NB REMEMBER resampling

/var/folders/gg/gk923hkx2w3bw72pk2shplydry9j0b/T/ipykernel_56629/3443846035.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)


### 2.1 Using BOW in Python
Start by importing a `CountVectorizer` object at the top of your notebook:

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

Let's instantiate a CountVectorizer object. Note that we set `lowercase = True` as we did not do this preprocessing ourselves :)

In [ ]:
vectorizer = CountVectorizer(lowercase=True)

We select our text column `"generation"` from our training split and use the `vectorizer` with the method `.fit_transform`:

In [23]:
X_train_bow = vectorizer.fit_transform(train_df["generation"])

For our validation split, we'll use `transform` only as we only `fit` our vectorizer to the training data!

In [22]:
X_val_bow = vectorizer.fit_transform(val_df["generation"])

Let's print the first few features and the first 10 vectors

In [ ]:
print("Features:", vectorizer.get_feature_names_out()[:10])

print("\nTraining set:")
for row in X_train_bow.toarray()[:10]:
    print(row)

Features: ['00' '000' '0001' '000m' '000th' '001' '004' '0051' '007' '00g']

Training set:
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]


As you can see, there are many features that are rarely present (the many 0's). It may be that the feature `00` is only used in one document. 

> Note: We could avoid this by setting a threshold `df_min` ([docs](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)). We won't for now.

### 2.2 Using TF-IDF in Python
The same steps can be used to vectorize text with `TF-IDF` with the only exception being the `vectorizer` that has changed!

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(lowercase=True)

### Your Turn: Defining a Vectorization Function!
In a second, we'll be seeing if using `bow` versus `tf-idf` will impact classification accuracy. For this, it would be nice to have a function that can do our workflow easily!

:::{admonition} HANDS-ON
:class: red
Define a function called `vectorize`. It should:
1. Have parameters`X_train` and `X_val` which can take text columns (e.g., `X_train = train_df["generations"]`).
2. Have a parameter called `vec_type` where you can choose either `"bow"` or `"tf-idf"` and it will select `CountVectorizer` and `TfidfVectorizer` respectively. You can do this with an `if` and `elif`statement
3. Return `X_train_vectorized` and `X_val_vectorized` 
:::

#### Solution
:::{admonition} HINT: `if` and `elif`? What do you mean!
:class: tip, dropdown
Remember how we can check a condition with `if`?

Let's say we want to print all banned colors!
```python
all_colors = ["red", "green", "blue", "yellow", "purple"]
banned_colors = ["red", "purple"]

# for each colour in the list called colours
for color in all_colors:
    if color in banned_colors: #  "in" checks if color is in banned_colors
        print(color + " is a BANNED color") # prints for red and purple
```

But we also want to print `yellow` because it is BRIGHT! We can do this with `elif`!
We can build on this with `elif`. Let's say that we want to ensure that people understand that the color `yellow` is a BRIGHT color:

```python
all_colors = ["red", "green", "blue", "yellow", "purple"]
banned_colors = ["red", "purple"]

# for each colour in the list called colours
for color in all_colors:
    if color in banned_colors: #  "in" checks if color is in banned_colors
        print(color + " is a BANNED color") # prints for red and purple
    elif color == "yellow": 
        print(color + " is a BRIGHT color")
```

**BONUS** If we want to catch everything that does not match either one of the conditions defined by `if` and `elif`, we can write `else` as the final statement!
```python
all_colors = ["red", "green", "blue", "yellow", "purple"]
banned_colors = ["red", "purple"]

# for each colour in the list called colours
for color in all_colors:
    if color in banned_colors: #  "in" checks if color is in banned_colors
        print(color + " is a BANNED color") # prints for red and purple
    elif color == "yellow": 
        print(color + " is a BRIGHT color")
    else: 
        # if none of the above
        print(color + " is allowed")  # prints for green and blue
```

:::

You can check the solution here:

In [ ]:
all_colors = ["red", "green", "blue", "yellow", "purple"]
banned_colors = ["red", "purple"]

# for each colour in the list called colours
for color in all_colors:
    if color in banned_colors: #  "in" checks if color is in banned_colors
        print(color + " is a BANNED color") # only prints: red and purple
    elif color == "yellow": 
        print(color + " is a BRIGHT color")
    else: 
        # if none of the above
        print(f"{color} is allowed")  # prints for green and blue

redis a BANNED color
green is allowed
blue is allowed
yellow is a BRIGHT color
purpleis a BANNED color


In [28]:
from typing import Literal 
# using Literal is not strictly necessary, 
# but is a way to define the options that you can use for the function!

def vectorize(X_train: pd.Series, X_val: pd.Series, vec_type:Literal["bow", "tf-idf"]):
    """
    Function to vectorize train and val data! 
    """
    if vec_type == "bow":
        vectorizer = CountVectorizer()
    elif vec_type== "tf-idf":
        vectorizer = TfidfVectorizer()
    else: 
        # this is good code practice, but if your function has no 'else' statement, this is also fine!
        raise ValueError(f"Invalid vec_type: {vec_type}. Must be either 'bow' or 'tf-idf")
    
    X_train_vectorized = vectorizer.fit_transform(X_train)
    X_val_vectorized = vectorizer.transform(X_val)

    return X_train_vectorized, X_val_vectorized